# BanglaBERT Implementation

In [1]:
#This prevents performance fluctuation
import random
import numpy as np
import torch

# ----------------------------
# Fix random seed for reproducibility
# ----------------------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Random seeds initialized")

DONE performace fluctuation


In [2]:
!pip uninstall -y transformers tokenizers huggingface_hub peft accelerate
!pip install --no-cache-dir transformers==4.40.2 tokenizers==0.19.1 huggingface_hub==0.23.0 peft==0.10.0 accelerate==0.28.0
print("Transformers and dependencies reinstalled.")

Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: accelerate 0.28.0
Uninstalling accelerate-0.28.0:
  Successfully uninstalled accelerate-0.28.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 138.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 248.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 372.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/19

In [3]:
!pip install transformers datasets torch scikit-learn
print("Core ML libraries installed.")

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.23.0
    Uninstalling huggingface-hub-0.23.0:
      Successfully uninstalled huggingface-hub-0.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
DONE torch


In [4]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/your train dataset location")
val_df = pd.read_csv("/kaggle/input/datasets/your validation dataset location")
test_df = pd.read_csv("/kaggle/input/datasets/your test dataset location")
print("DataFrames loaded successfully.")

DONE locating dataset


In [5]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
print("Input files listed.")

/kaggle/input/datasets/avrawdas/fake-news-dataset/validation.csv
/kaggle/input/datasets/avrawdas/fake-news-dataset/train.csv
/kaggle/input/datasets/avrawdas/fake-news-dataset/test.csv
/kaggle/input/datasets/avrawdas/my-trained-banglabert/__huggingface_repos__.json
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/config.json
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/trainer_state.json
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/training_args.bin
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/tokenizer.json
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/tokenizer_config.json
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/scheduler.pt
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint-8085/model.safetensors
/kaggle/input/datasets/avrawdas/my-trained-banglabert/results/checkpoint

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")
print("Bangla BERT tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

DONE bert


In [7]:
def tokenize(data):
    return tokenizer(
        data['Content'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

print("Tokenization function defined.")

DONE tokenizing


In [8]:
from datasets import Dataset

# Step 1: Extract from pandas using the exact name it currently has (Capital 'L')
train_dataset = Dataset.from_pandas(train_df[['Content', 'Label']])
val_dataset = Dataset.from_pandas(val_df[['Content', 'Label']])
test_dataset = Dataset.from_pandas(test_df[['Content', 'Label']])

# Step 2: Rename the column to lowercase 'label' for the model's training phase
train_dataset = train_dataset.rename_column("Label", "label")
val_dataset = val_dataset.rename_column("Label", "label")
test_dataset = test_dataset.rename_column("Label", "label")
print("Dataset schema aligned (Label -> label).")

DONE label


In [9]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
print("Dataset encoded and batched.")

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/43106 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/9237 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [1015, 1016, 1017, 1018, 1053, 1054, 1055, 1056, 1057, 1060, 1061, 1062, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1099, 1101, 1112, 1113, 1556, 1557, 1568], your vocabulary could be corrupted !


Map:   0%|          | 0/9238 [00:00<?, ? examples/s]

DONE dataset load


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "sagorsarker/bangla-bert-base",
    num_labels=2
)
print("Model architecture initialized.")

In [ ]:
model.config.hidden_dropout_prob = 0.3
model.config.attention_probs_dropout_prob = 0.3
print("Regularization parameters adjusted.")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,                # your LR bump
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,                # 3 or 4 epochs
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    lr_scheduler_type="linear",
    warmup_ratio=0.1
)
print("Training hyperparameters and evaluation strategy defined.")

In [ ]:
from transformers import EarlyStoppingCallback
print("Early stopping criteria initialized.")

In [ ]:
from transformers import Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Define the metric calculation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# 2. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  # Stops if val_loss rises
)
print("Model training pipeline initialized.")

In [ ]:
import wandb

# Log in using your specific key
wandb.login(key="Your api key from wandb")
print("Weights & Biases authentication successful.")

In [ ]:
# Make sure your Trainer object is named trainer
trainer.train()
print("Model fine-tuning complete.")

# Embedding

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

# 1. Paste the path you just copied right here!
save_directory = "/kaggle/input/datasets/gob002/my-trained-banglabert/my_saved_model"

# 2. Load your pre-trained model directly from the dataset
print("Loading model from dataset...")
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

# 3. Move it to the GPU and set to evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model successfully loaded and ready for embedding extraction!")

In [ ]:
import torch
from torch.utils.data import DataLoader

# 1. Define the Dataloader function
def create_dataloader(dataset, batch_size=16):
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return DataLoader(dataset, batch_size=batch_size)

# 2. Define the Extraction function
def get_embeddings(dataloader):
    all_embeddings = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Pass through the base model
            outputs = model.base_model(input_ids=input_ids, attention_mask=attention_mask)
            
            # Get the [CLS] token embeddings
            cls_embeddings = outputs.last_hidden_state[:, 0, :] 
            all_embeddings.append(cls_embeddings.cpu())
            
    return torch.cat(all_embeddings, dim=0)

print("Bridge functions defined into memory!")

In [ ]:
# 1. Create the Loaders (Using your existing create_dataloader function)
train_loader = create_dataloader(train_dataset)
val_loader = create_dataloader(val_dataset)
test_loader = create_dataloader(test_dataset)

# 2. Extract using the LOADERS, not the datasets
print("Starting extraction (this might take a few minutes)...")
train_embeddings = get_embeddings(train_loader)
val_embeddings = get_embeddings(val_loader)
test_embeddings = get_embeddings(test_loader)

print(f"Extraction Complete! Train Shape: {train_embeddings.shape}")

In [ ]:
# Save the math (Embeddings)
torch.save(train_embeddings, "train_embeddings.pt")
torch.save(val_embeddings, "val_embeddings.pt")
torch.save(test_embeddings, "test_embeddings.pt")

# Save the answers (Labels) - Your GAT needs these!
torch.save(torch.tensor(train_dataset['label']), "train_labels.pt")
torch.save(torch.tensor(val_dataset['label']), "val_labels.pt")
torch.save(torch.tensor(test_dataset['label']), "test_labels.pt")

print("ALL FILES (.pt) ARE SAVED IN /kaggle/working/")

# GAT Implementation

In [ ]:
!pip install torch-geometric
print("PyTorch Geometric (PyG) installed.")

In [ ]:
# 1. Install PyTorch Geometric (Colab needs this installed every time it starts)
!pip install torch-geometric

# 2. Import the necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.metrics import accuracy_score, f1_score

print("Libraries installed and imported!")

In [ ]:
print("Loading saved Phase 1 data...")

# Load the features (The 768-dimensional math from BanglaBERT)
train_embeddings = torch.load("/kaggle/working/train_embeddings.pt")
val_embeddings = torch.load("/kaggle/working/val_embeddings.pt")
test_embeddings = torch.load("/kaggle/working/test_embeddings.pt")

# Load the answers (0 for Real, 1 for Fake)
train_y = torch.load("/kaggle/working/train_labels.pt")
val_y = torch.load("/kaggle/working/val_labels.pt")
test_y = torch.load("/kaggle/working/test_labels.pt")

print("Data successfully loaded!")
print(f"Train nodes: {train_embeddings.shape[0]}")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data

print("Building smart similarity graphs (5 neighbors)...")

def create_knn_edges(x, k):
    similarity = torch.mm(x, x.t())
    similarity.fill_diagonal_(-float('inf'))
    _, top_k_indices = torch.topk(similarity, k=k, dim=1)
    src = torch.arange(x.size(0), device=x.device).repeat_interleave(k)
    dst = top_k_indices.flatten()
    return torch.stack([src, dst], dim=0)

# Normalize and build
train_emb_norm = F.normalize(train_embeddings.float(), p=2, dim=1)
val_emb_norm = F.normalize(val_embeddings.float(), p=2, dim=1)
test_emb_norm = F.normalize(test_embeddings.float(), p=2, dim=1)

k_edges = 5
train_edges = create_knn_edges(train_emb_norm, k_edges)
val_edges = create_knn_edges(val_emb_norm, k_edges)
test_edges = create_knn_edges(test_emb_norm, k_edges)

train_data = Data(x=train_embeddings.float(), edge_index=train_edges, y=train_y)
val_data   = Data(x=val_embeddings.float(), edge_index=val_edges, y=val_y)
test_data  = Data(x=test_embeddings.float(), edge_index=test_edges, y=test_y)

print("Smart Graphs built instantly!")

In [ ]:
class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8):
        super(GAT, self).__init__()
        
        # First Attention Layer: Added edge_dim=1 to accept Cosine Similarity scores
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=0.6, edge_dim=1)
        
        # Output Layer: Added edge_dim=1 here as well
        self.gat2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False, dropout=0.6, edge_dim=1)

    # Added edge_attr to the forward pass
    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.6, training=self.training)
        
        # Pass the edge weights (edge_attr) into the attention mechanism
        x = F.elu(self.gat1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        
        return x

print("UPGRADED GAT Architecture defined (Now with Edge Weights!)")

In [ ]:
import torch.nn.functional as F

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# 2. Define the Focal Loss Mathematical Function
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        return F_loss.mean()

# 3. Initialize Model and Optimizer (with Hyperparameter Upgrades)
gat_model = GAT(in_dim=768, hidden_dim=256, out_dim=2, heads=8).to(device)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.001, weight_decay=5e-4)

# 4. Initialize the NEW Focal Loss Function
criterion = FocalLoss(alpha=1, gamma=2).to(device)

# 5. Move the newly weighted graphs to the GPU
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

print("Upgraded Model, Focal Loss, and Weighted Data ready on GPU!")

In [ ]:
def train():
    gat_model.train()
    optimizer.zero_grad()
    # FIX: Added train_data.edge_attr right here!
    out = gat_model(train_data.x, train_data.edge_index, train_data.edge_attr)
    loss = criterion(out, train_data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(data):
    gat_model.eval()
    with torch.no_grad():
        # FIX: Added data.edge_attr right here!
        out = gat_model(data.x, data.edge_index, data.edge_attr)
        preds = out.argmax(dim=1).cpu().numpy()
        labels = data.y.cpu().numpy()
        
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='macro')
    return acc, f1

# Run the Training Loop
print("Starting GAT Training with Early Stopping...")

best_val_f1 = 0
best_epoch = 0

for epoch in range(1, 151):
    loss = train()
    if epoch % 10 == 0:
        val_acc, val_f1 = evaluate(val_data)
        print(f'Epoch {epoch:03d} | Train Loss: {loss:.4f} | Val Accuracy: {val_acc:.4f} | Val F1: {val_f1:.4f}')
        
        # Save the absolute best iteration of the model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            torch.save(gat_model.state_dict(), 'sota_model.pth')
            print(f"   New best model saved at Epoch {epoch}!")

print("========================================")
print(f"Training Complete. Best model was from Epoch {best_epoch}.")
print("========================================")

In [ ]:
# Evaluate on the completely unseen TEST set
print("Evaluating Final Model on the Test Set...")

test_acc, test_f1 = evaluate(test_data)

print("========================================")
print("OFFICIAL FINAL RESULTS FOR THESIS 🎉")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print("========================================")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch

# Get the final predictions from the test set
gat_model.eval()
with torch.no_grad():
    try:
        # Try the stable baseline method first (2 arguments)
        out = gat_model(test_data.x, test_data.edge_index)
    except TypeError:
        # If the model is the advanced version and demands weights, use 3 arguments!
        out = gat_model(test_data.x, test_data.edge_index, test_data.edge_attr)
        
    preds = out.argmax(dim=1).cpu().numpy()
    labels = test_data.y.cpu().numpy()

# Create the confusion matrix
cm = confusion_matrix(labels, preds)

# Plot it beautifully
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real News', 'Fake News'], 
            yticklabels=['Real News', 'Fake News'],
            annot_kws={"size": 16})

plt.title('Final Model Confusion Matrix', fontsize=16)
plt.ylabel('Actual Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()